In [3]:
import numpy as np


def objective_function(x):
    """Objective function to minimize: f(x) = x^2 - 6x - 13"""
    return x**2 - 6 * x - 13


def simulated_annealing():
    # Initial Settings
    x_current = 1
    T = 10.0
    alpha = 0.5
    iterations = 4

    # Provided random numbers for accepting worse solutions
    random_numbers = [0.30, 0.80, 0.20, 0.90]
    r_idx = 0

    # Best solution tracking
    x_best = x_current
    f_best = objective_function(x_best)

    print(
        f"{'Iter':<5} | {'T':<6} | {'x':<5} | {'x_prime':<7} | {'f(x)':<7} | {'f(x_prime)':<10} | {'df':<5} | {'P':<7} | {'r':<5} | {'Decision':<8} | {'Best x':<6}"
    )
    print("-" * 95)

    for i in range(1, iterations + 1):
        f_current = objective_function(x_current)

        # Neighbour generation: x' = x + 1
        x_prime = x_current + 1
        f_prime = objective_function(x_prime)

        # Change in objective function value
        delta_f = f_prime - f_current

        r_str = "N/A"

        # Decision Rule
        if delta_f <= 0:
            P = 1.0
            decision = "Accept"
            x_current = x_prime
        else:
            P = np.exp(-delta_f / T)
            r = random_numbers[r_idx]
            r_idx += 1
            r_str = f"{r:.2f}"

            if r <= P:
                decision = "Accept"
                x_current = x_prime
            else:
                decision = "Reject"

        # Update best solution found so far
        if objective_function(x_current) < f_best:
            x_best = x_current
            f_best = objective_function(x_best)

        # Print iteration metrics
        print(
            f"{i:<5} | {T:<6.2f} | {x_current if decision == 'Reject' else x_prime - 1:<5} | {x_prime:<7} | {f_current:<7} | {f_prime:<10} | {delta_f:<5} | {P:<7.4f} | {r_str:<5} | {decision:<8} | {x_best:<6}"
        )

        # Cooling schedule update for next iteration
        T = T * alpha

    print("-" * 95)
    print(f"Optimal solution found: x = {x_best}, f(x) = {f_best}")


if __name__ == "__main__":
    simulated_annealing()

Iter  | T      | x     | x_prime | f(x)    | f(x_prime) | df    | P       | r     | Decision | Best x
-----------------------------------------------------------------------------------------------
1     | 10.00  | 1     | 2       | -18     | -21        | -3    | 1.0000  | N/A   | Accept   | 2     
2     | 5.00   | 2     | 3       | -21     | -22        | -1    | 1.0000  | N/A   | Accept   | 3     
3     | 2.50   | 3     | 4       | -22     | -21        | 1     | 0.6703  | 0.30  | Accept   | 3     
4     | 1.25   | 4     | 5       | -21     | -18        | 3     | 0.0907  | 0.80  | Reject   | 3     
-----------------------------------------------------------------------------------------------
Optimal solution found: x = 3, f(x) = -22


In [4]:
import numpy as np

# --- Problem Formulation & Setup ---
# Objective Function: Maximize f(x) = -x^2 + 2x, where 0 <= x <= 2
def f(x):
    return -x**2 + 2*x

# Decode binary chromosome string to real number in [0, 2]
def decode(chromosome, x_min=0.0, x_max=2.0):
    bits = len(chromosome)
    decimal_val = int(chromosome, 2)
    return x_min + (decimal_val / (2**bits - 1)) * (x_max - x_min)

def genetic_algorithm_step():
    # Given initial population
    population = ["11010", "00111", "10110", "00101"]
    random_nums = [0.4, 0.15, 0.7, 0.9]
    
    print("=== Step 1: Initial Population & Fitness Evaluation ===")
    decoded_x = [decode(chrom) for chrom in population]
    fitness_vals = [f(x) for x in decoded_x]
    
    for i, (chrom, x, fit) in enumerate(zip(population, decoded_x, fitness_vals), 1):
        print(f"Chromosome {i}: {chrom} | x = {x:.4f} | f(x) = {fit:.4f}")
        
    total_fitness = sum(fitness_vals)
    probs = [fit / total_fitness for fit in fitness_vals]
    cum_probs = np.cumsum(probs)
    
    print("\n=== Step 2: Selection (Roulette Wheel Selection) ===")
    print(f"Total Fitness = {total_fitness:.4f}")
    for i, (p, cp) in enumerate(zip(probs, cum_probs), 1):
        print(f"Chrom {i} Probability: {p:.4f} | Cumulative Probability: {cp:.4f}")
        
    # Mating Pool Selection using given random numbers
    mating_pool = []
    print("\nSelecting Parents using given random numbers:")
    for r in random_nums:
        for idx, cp in enumerate(cum_probs):
            if r <= cp:
                mating_pool.append(population[idx])
                print(f"Random No. {r} -> Selects Chromosome {idx + 1} ({population[idx]})")
                break

    print("\n=== Step 3: Single-Point Crossover ===")
    # Pair 1: Parent 1 & Parent 2 with Crossover Point after 1st digit
    p1, p2 = mating_pool[0], mating_pool[1]
    cut1 = 1
    o1 = p1[:cut1] + p2[cut1:]
    o2 = p2[:cut1] + p1[cut1:]
    print(f"Pair 1 (Crossover Point = {cut1}):")
    print(f"  Parents:   {p1} and {p2}")
    print(f"  Offspring: {o1} and {o2}")

    # Pair 2: Parent 3 & Parent 4 with Crossover Point before 5th digit (after 4th digit)
    p3, p4 = mating_pool[2], mating_pool[3]
    cut2 = 4
    o3 = p3[:cut2] + p4[cut2:]
    o4 = p4[:cut2] + p3[cut2:]
    print(f"\nPair 2 (Crossover Point = {cut2} [before 5th digit]):")
    print(f"  Parents:   {p3} and {p4}")
    print(f"  Offspring: {o3} and {o4}")

    # New Population Evaluation
    new_population = [o1, o2, o3, o4]
    print("\n=== Step 4: New Generation Evaluation ===")
    for i, chrom in enumerate(new_population, 1):
        x = decode(chrom)
        fit = f(x)
        print(f"New Chromosome {i}: {chrom} | x = {x:.4f} | f(x) = {fit:.4f}")

if __name__ == "__main__":
    genetic_algorithm_step()

=== Step 1: Initial Population & Fitness Evaluation ===
Chromosome 1: 11010 | x = 1.6774 | f(x) = 0.5411
Chromosome 2: 00111 | x = 0.4516 | f(x) = 0.6993
Chromosome 3: 10110 | x = 1.4194 | f(x) = 0.8241
Chromosome 4: 00101 | x = 0.3226 | f(x) = 0.5411

=== Step 2: Selection (Roulette Wheel Selection) ===
Total Fitness = 2.6056
Chrom 1 Probability: 0.2077 | Cumulative Probability: 0.2077
Chrom 2 Probability: 0.2684 | Cumulative Probability: 0.4760
Chrom 3 Probability: 0.3163 | Cumulative Probability: 0.7923
Chrom 4 Probability: 0.2077 | Cumulative Probability: 1.0000

Selecting Parents using given random numbers:
Random No. 0.4 -> Selects Chromosome 2 (00111)
Random No. 0.15 -> Selects Chromosome 1 (11010)
Random No. 0.7 -> Selects Chromosome 3 (10110)
Random No. 0.9 -> Selects Chromosome 4 (00101)

=== Step 3: Single-Point Crossover ===
Pair 1 (Crossover Point = 1):
  Parents:   00111 and 11010
  Offspring: 01010 and 10111

Pair 2 (Crossover Point = 4 [before 5th digit]):
  Parents:   

In [5]:
import numpy as np


def fitness(x):
    """Fitness function: F(x) = x^2"""
    return x**2


def decode_chromosome(binary_str):
    """Converts a 5-bit binary string directly to its integer x-value."""
    return int(binary_str, 2)


def run_genetic_algorithm():
    # Problem Specifications from Image
    # Range: 0 < x < 31
    initial_population = ["11011", "10001", "01111", "10111"]

    print("=== Step 1: Initial Population & Fitness Evaluation ===")
    x_values = [decode_chromosome(chrom) for chrom in initial_population]
    fitness_values = [fitness(x) for x in x_values]

    for i, (chrom, x, fit) in enumerate(
        zip(initial_population, x_values, fitness_values), 1
    ):
        print(
            f"String {i}: Chromosome = {chrom} | x-value = {x} | Fitness F(x) = {fit}"
        )

    # Calculate probabilities for Roulette Wheel Selection
    total_fitness = sum(fitness_values)
    probs = [fit / total_fitness for fit in fitness_values]
    cum_probs = np.cumsum(probs)

    print("\n=== Step 2: Selection Probabilities ===")
    print(f"Total Fitness sum = {total_fitness}")
    for i, (p, cp) in enumerate(zip(probs, cum_probs), 1):
        print(
            f"String {i}: Selection Probability = {p:.4f} | Cumulative Probability = {cp:.4f}"
        )

    # Example 1-generation iteration using single-point crossover
    print("\n=== Step 3: Single-Point Crossover (Example Iteration) ===")

    # Select pairs: (1, 2) and (3, 4) with crossover point = 2
    crossover_point = 2

    # Pair 1: String 1 & String 2
    p1, p2 = initial_population[0], initial_population[1]
    o1 = p1[:crossover_point] + p2[crossover_point:]
    o2 = p2[:crossover_point] + p1[crossover_point:]

    # Pair 2: String 3 & String 4
    p3, p4 = initial_population[2], initial_population[3]
    o3 = p3[:crossover_point] + p4[crossover_point:]
    o4 = p4[:crossover_point] + p3[crossover_point:]

    new_population = [o1, o2, o3, o4]

    print(
        f"Offspring Pair 1 (Cut at {crossover_point}): {p1} + {p2} -> {o1}, {o2}"
    )
    print(
        f"Offspring Pair 2 (Cut at {crossover_point}): {p3} + {p4} -> {o3}, {o4}"
    )

    print("\n=== Step 4: Next Generation Evaluation ===")
    new_x = [decode_chromosome(chrom) for chrom in new_population]
    new_fitness = [fitness(x) for x in new_x]

    for i, (chrom, x, fit) in enumerate(
        zip(new_population, new_x, new_fitness), 1
    ):
        print(
            f"New String {i}: Chromosome = {chrom} | x-value = {x} | Fitness F(x) = {fit}"
        )

    best_idx = np.argmax(new_fitness)
    print(
        f"\nMaximum value in generation: x = {new_x[best_idx]} with F(x) = {new_fitness[best_idx]}"
    )


if __name__ == "__main__":
    run_genetic_algorithm()

=== Step 1: Initial Population & Fitness Evaluation ===
String 1: Chromosome = 11011 | x-value = 27 | Fitness F(x) = 729
String 2: Chromosome = 10001 | x-value = 17 | Fitness F(x) = 289
String 3: Chromosome = 01111 | x-value = 15 | Fitness F(x) = 225
String 4: Chromosome = 10111 | x-value = 23 | Fitness F(x) = 529

=== Step 2: Selection Probabilities ===
Total Fitness sum = 1772
String 1: Selection Probability = 0.4114 | Cumulative Probability = 0.4114
String 2: Selection Probability = 0.1631 | Cumulative Probability = 0.5745
String 3: Selection Probability = 0.1270 | Cumulative Probability = 0.7015
String 4: Selection Probability = 0.2985 | Cumulative Probability = 1.0000

=== Step 3: Single-Point Crossover (Example Iteration) ===
Offspring Pair 1 (Cut at 2): 11011 + 10001 -> 11001, 10011
Offspring Pair 2 (Cut at 2): 01111 + 10111 -> 01111, 10111

=== Step 4: Next Generation Evaluation ===
New String 1: Chromosome = 11001 | x-value = 25 | Fitness F(x) = 625
New String 2: Chromosome = 